In [1]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn import decomposition
from scipy import linalg
import matplotlib.pyplot as plt

%matplotlib inline
np.set_printoptions(suppress=True)

In [2]:
categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space']
remove = ('headers', 'footers', 'quotes')
newsgroups_train = fetch_20newsgroups(subset='train', categories=categories, remove=remove)
newsgroups_test = fetch_20newsgroups(subset='test', categories=categories, remove=remove)

newsgroups_train.filenames.shape, newsgroups_train.target.shape

((2034,), (2034,))

In [3]:
print("\n".join(newsgroups_train.data[:3]))

Hi,

I've noticed that if you only save a model (with all your mapping planes
positioned carefully) to a .3DS file that when you reload it after restarting
3DS, they are given a default position and orientation.  But if you save
to a .PRJ file their positions/orientation are preserved.  Does anyone
know why this information is not stored in the .3DS file?  Nothing is
explicitly said in the manual about saving texture rules in the .PRJ file. 
I'd like to be able to read the texture rule information, does anyone have 
the format for the .PRJ file?

Is the .CEL file format available from somewhere?

Rych


Seems to be, barring evidence to the contrary, that Koresh was simply
another deranged fanatic who thought it neccessary to take a whole bunch of
folks with him, children and all, to satisfy his delusional mania. Jim
Jones, circa 1993.


Nope - fruitcakes like Koresh have been demonstrating such evil corruption
for centuries.

 >In article <1993Apr19.020359.26996@sq.sq.com>, msb@sq.sq.c

In [4]:
np.array(newsgroups_train.target_names)[newsgroups_train.target[:3]]

array(['comp.graphics', 'talk.religion.misc', 'sci.space'], dtype='<U18')

In [5]:
newsgroups_train.target[:10]

array([1, 3, 2, 0, 2, 0, 2, 1, 2, 1], dtype=int64)

In [6]:
num_topics, num_top_words = 6, 8

In [7]:
from sklearn.feature_extraction import _stop_words

sorted(list(_stop_words.ENGLISH_STOP_WORDS))[:20]

['a',
 'about',
 'above',
 'across',
 'after',
 'afterwards',
 'again',
 'against',
 'all',
 'almost',
 'alone',
 'along',
 'already',
 'also',
 'although',
 'always',
 'am',
 'among',
 'amongst',
 'amoungst']

In [8]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Sandipan.Dey\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [9]:
from nltk import stem
wnl = stem.WordNetLemmatizer()
porter = stem.porter.PorterStemmer()
word_list = ['feet', 'foot', 'foots', 'footing']
[wnl.lemmatize(word) for word in word_list]

['foot', 'foot', 'foot', 'footing']

In [10]:
[porter.stem(word) for word in word_list]

['feet', 'foot', 'foot', 'foot']

In [14]:
import spacy
from spacy.lemmatizer import Lemmatizer
#lemmatizer = Lemmatizer()
#[lemmatizer.lookup(word) for word in word_list]

In [15]:
#pip install spacy==2.3.5
#!python -m spacy download en_core_web_sm

In [16]:
nlp = spacy.load("en_core_web_sm")
sorted(list(nlp.Defaults.stop_words))[:20]

["'d",
 "'ll",
 "'m",
 "'re",
 "'s",
 "'ve",
 'a',
 'about',
 'above',
 'across',
 'after',
 'afterwards',
 'again',
 'against',
 'all',
 'almost',
 'alone',
 'along',
 'already',
 'also']

In [17]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import nltk
vectorizer = CountVectorizer(stop_words='english') #, tokenizer=LemmaTokenizer())
vectors = vectorizer.fit_transform(newsgroups_train.data).todense() # (documents, vocab)
vectors.shape #, vectors.nnz / vectors.shape[0], row_means.shape

(2034, 26576)

In [18]:
print(len(newsgroups_train.data), vectors.shape)
vocab = np.array(vectorizer.get_feature_names())
vocab.shape

2034 (2034, 26576)


C:\Users\Sandipan.Dey\anaconda3\lib\site-packages\sklearn\utils\deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


(26576,)

In [19]:
vocab[7000:7020]

array(['cosmonauts', 'cosmos', 'cosponsored', 'cost', 'costa', 'costar',
       'costing', 'costly', 'costruction', 'costs', 'cosy', 'cote',
       'couched', 'couldn', 'council', 'councils', 'counsel',
       'counselees', 'counselor', 'count'], dtype='<U80')

In [20]:
%time U, s, Vh = linalg.svd(vectors, full_matrices=False)

Wall time: 1min 54s


In [21]:
print(U.shape, s.shape, Vh.shape)

(2034, 2034) (2034,) (2034, 26576)


In [22]:
s[:4]

array([433.92698542, 291.51012741, 240.71137677, 220.00048043])

In [23]:
np.diag(np.diag(s[:4]))

array([433.92698542, 291.51012741, 240.71137677, 220.00048043])

In [24]:
num_top_words=8

def show_topics(a):
    top_words = lambda t: [vocab[i] for i in np.argsort(t)[:-num_top_words-1:-1]]
    topic_words = ([top_words(t) for t in a])
    return [' '.join(t) for t in topic_words]

show_topics(Vh[:10])

['critus ditto propagandist surname galacticentric kindergarten surreal imaginative',
 'jpeg gif file color quality image jfif format',
 'graphics edu pub mail 128 3d ray ftp',
 'jesus god matthew people atheists atheism does graphics',
 'image data processing analysis software available tools display',
 'god atheists atheism religious believe religion argument true',
 'space nasa lunar mars probe moon missions probes',
 'image probe surface lunar mars probes moon orbit',
 'argument fallacy conclusion example true ad argumentum premises',
 'space larson image theory universe physical nasa material']

In [25]:
m,n=vectors.shape
d=5  # num topics
clf = decomposition.NMF(n_components=d, random_state=1)

W1 = clf.fit_transform(vectors)
H1 = clf.components_

show_topics(H1)

C:\Users\Sandipan.Dey\anaconda3\lib\site-packages\sklearn\utils\validation.py:598: FutureWarning: np.matrix usage is deprecated in 1.0 and will raise a TypeError in 1.2. Please convert to a numpy array with np.asarray. For more information see: https://numpy.org/doc/stable/reference/generated/numpy.matrix.html
  FutureWarning,
C:\Users\Sandipan.Dey\anaconda3\lib\site-packages\sklearn\decomposition\_nmf.py:294: FutureWarning: The 'init' value, when 'init=None' and n_components is less than n_samples and n_features, will be changed from 'nndsvd' to 'nndsvda' in 1.1 (renaming of 0.26).
  FutureWarning,


['jpeg image gif file color images format quality',
 'edu graphics pub mail 128 ray ftp send',
 'space launch satellite nasa commercial satellites year market',
 'jesus god people matthew atheists does atheism said',
 'image data available software processing ftp edu analysis']

In [26]:
vectorizer_tfidf = TfidfVectorizer(stop_words='english')
vectors_tfidf = vectorizer_tfidf.fit_transform(newsgroups_train.data) # (documents, vocab)
newsgroups_train.data[10:20]
W1 = clf.fit_transform(vectors_tfidf)
H1 = clf.components_
show_topics(H1)

C:\Users\Sandipan.Dey\anaconda3\lib\site-packages\sklearn\decomposition\_nmf.py:294: FutureWarning: The 'init' value, when 'init=None' and n_components is less than n_samples and n_features, will be changed from 'nndsvd' to 'nndsvda' in 1.1 (renaming of 0.26).
  FutureWarning,


['people don think just like objective say morality',
 'graphics thanks files image file program windows know',
 'space nasa launch shuttle orbit moon lunar earth',
 'ico bobbe tek beauchaine bronx manhattan sank queens',
 'god jesus bible believe christian atheism does belief']

In [27]:
%time u, s, v = np.linalg.svd(vectors, full_matrices=False)
from sklearn import decomposition
import fbpca
%time u, s, v = decomposition.randomized_svd(vectors, 10)
%time u, s, v = fbpca.pca(vectors, 10)

Wall time: 2min 7s


ModuleNotFoundError: No module named 'fbpca'

In [28]:
import gensim
corpus = "In terms of unforgettable looks, and enduring desire from enthusiasts who may have grown up gluing together the AMT 3-in-1 model kit that it inspired, the 1940 models stand today as some of the most iconic, instantly recognizable automobiles that the Ford Motor Company ever produced. That year, Fords were produced in two series: Standard and Deluxe. The easiest way to tell them apart is to look for a cleaner one-piece grille on Standard models, while the Deluxe version has a three-piece grille assembly. Both cars also had slightly different pieces of hood trim. This 1940 Ford Standard Tudor sedan was a very popular model that year–around 151,000 of them were built and sold. This Standard has been under the same California ownership since 1994, after the seller bought it from an owner in Texas. The seller describes the car as being entirely original, though the age of the finish and status of any restoration or refresh are unknown."
from nltk import sent_tokenize
list_of_sentence = sent_tokenize(corpus)
list_of_sentence

['In terms of unforgettable looks, and enduring desire from enthusiasts who may have grown up gluing together the AMT 3-in-1 model kit that it inspired, the 1940 models stand today as some of the most iconic, instantly recognizable automobiles that the Ford Motor Company ever produced.',
 'That year, Fords were produced in two series: Standard and Deluxe.',
 'The easiest way to tell them apart is to look for a cleaner one-piece grille on Standard models, while the Deluxe version has a three-piece grille assembly.',
 'Both cars also had slightly different pieces of hood trim.',
 'This 1940 Ford Standard Tudor sedan was a very popular model that year–around 151,000 of them were built and sold.',
 'This Standard has been under the same California ownership since 1994, after the seller bought it from an owner in Texas.',
 'The seller describes the car as being entirely original, though the age of the finish and status of any restoration or refresh are unknown.']

In [29]:
list_of_simple_preprocess_data = []
for i in list_of_sentence:
    list_of_simple_preprocess_data.append(gensim.utils.simple_preprocess(i, deacc=True, min_len=3))
texts = list_of_simple_preprocess_data
texts

[['terms',
  'unforgettable',
  'looks',
  'and',
  'enduring',
  'desire',
  'from',
  'enthusiasts',
  'who',
  'may',
  'have',
  'grown',
  'gluing',
  'together',
  'the',
  'amt',
  'model',
  'kit',
  'that',
  'inspired',
  'the',
  'models',
  'stand',
  'today',
  'some',
  'the',
  'most',
  'iconic',
  'instantly',
  'recognizable',
  'automobiles',
  'that',
  'the',
  'ford',
  'motor',
  'company',
  'ever',
  'produced'],
 ['that',
  'year',
  'fords',
  'were',
  'produced',
  'two',
  'series',
  'standard',
  'and',
  'deluxe'],
 ['the',
  'easiest',
  'way',
  'tell',
  'them',
  'apart',
  'look',
  'for',
  'cleaner',
  'one',
  'piece',
  'grille',
  'standard',
  'models',
  'while',
  'the',
  'deluxe',
  'version',
  'has',
  'three',
  'piece',
  'grille',
  'assembly'],
 ['both',
  'cars',
  'also',
  'had',
  'slightly',
  'different',
  'pieces',
  'hood',
  'trim'],
 ['this',
  'ford',
  'standard',
  'tudor',
  'sedan',
  'was',
  'very',
  'popular',
  

In [30]:
bigram = gensim.models.Phrases(list_of_simple_preprocess_data) 
bigram

In [34]:
#from gensim.utils import lemmatize
from nltk.corpus import stopwords
stops = set(stopwords.words('english')) 

def process_texts(texts):
    texts = [[word for word in line if word not in stops] for line in texts]
    texts = [bigram[line] for line in texts]
    #texts = [[word.decode("utf-8").split('/')[0] for word in lemmatize(' '.join(line), allowed_tags=re.compile('(NN)'), min_length=5)] for line in texts]
    return texts

import re
train_texts = process_texts(list_of_simple_preprocess_data)

from gensim.models import LdaModel
#from gensim.models.wrappers import LdaMallet
from gensim.corpora import Dictionary

train_texts

[['terms',
  'unforgettable',
  'looks',
  'enduring',
  'desire',
  'enthusiasts',
  'may',
  'grown',
  'gluing',
  'together',
  'amt',
  'model',
  'kit',
  'inspired',
  'models',
  'stand',
  'today',
  'iconic',
  'instantly',
  'recognizable',
  'automobiles',
  'ford',
  'motor',
  'company',
  'ever',
  'produced'],
 ['year', 'fords', 'produced', 'two', 'series', 'standard', 'deluxe'],
 ['easiest',
  'way',
  'tell',
  'apart',
  'look',
  'cleaner',
  'one',
  'piece',
  'grille',
  'standard',
  'models',
  'deluxe',
  'version',
  'three',
  'piece',
  'grille',
  'assembly'],
 ['cars', 'also', 'slightly', 'different', 'pieces', 'hood', 'trim'],
 ['ford',
  'standard',
  'tudor',
  'sedan',
  'popular',
  'model',
  'year',
  'around',
  'built',
  'sold'],
 ['standard',
  'california',
  'ownership',
  'since',
  'seller',
  'bought',
  'owner',
  'texas'],
 ['seller',
  'describes',
  'car',
  'entirely',
  'original',
  'though',
  'age',
  'finish',
  'status',
  'rest

In [37]:
dictionary = Dictionary(train_texts)
print(dictionary)
print(corpus)
ldamodel = LdaModel(corpus=corpus, num_topics=2, id2word=dictionary)
ldamodel.show_topics()

Dictionary(75 unique tokens: ['amt', 'automobiles', 'company', 'desire', 'enduring']...)
In terms of unforgettable looks, and enduring desire from enthusiasts who may have grown up gluing together the AMT 3-in-1 model kit that it inspired, the 1940 models stand today as some of the most iconic, instantly recognizable automobiles that the Ford Motor Company ever produced. That year, Fords were produced in two series: Standard and Deluxe. The easiest way to tell them apart is to look for a cleaner one-piece grille on Standard models, while the Deluxe version has a three-piece grille assembly. Both cars also had slightly different pieces of hood trim. This 1940 Ford Standard Tudor sedan was a very popular model that year–around 151,000 of them were built and sold. This Standard has been under the same California ownership since 1994, after the seller bought it from an owner in Texas. The seller describes the car as being entirely original, though the age of the finish and status of any re

ValueError: not enough values to unpack (expected 2, got 1)